> Repository edition: run from the repository root using an environment with the dependencies in `requirements.txt`. Machine-specific environment paths have been replaced. Historical outputs are retained except environment checks; they are not evidence of a fresh rerun. See `docs/REPRODUCIBILITY.md`.


In [1]:
from pathlib import Path
import numpy as np

# Check the working directory
print("Working directory:", Path.cwd())

# Check the initial data and cumulative feedback files
print("Initial data folder found:", Path("initial_data").is_dir())
print("Cumulative inputs found:", Path("week_2/inputs.txt").is_file())
print("Cumulative outputs found:", Path("week_2/outputs.txt").is_file())

Working directory: c:\Users\AG\BeatrizGIT\Master\CapstoneProject
Initial data folder found: True
Cumulative inputs found: True
Cumulative outputs found: True


Here we check two things before starting the modelling process:

1. that the notebook is being executed from the correct project folder;
2. that the required initial-data files and previous-round files are available.

The output confirms:

- `Working directory`: we are inside `CapstoneProject`;
- `Initial data folder found: True`;
- `Week 1 inputs fo### 💬 Explanation — file and working-directory check

This cell checks that the notebook is running from the correct `CapstoneProject` directory and that the files required for **Round 3** are available.

The output confirms that:

- the working directory is correct;
- the `initial_data` folder exists;
- the cumulative `week_2/inputs.txt` file exists;
- the cumulative `week_2/outputs.txt` file exists.

These `week_2` files contain the submitted queries and returned outputs from the first two rounds. This check is important because all later cells use relative paths; if the notebook were launched from a different folder, the data-loading stage would fail.
und: True`;
- `Week 1 outputs found: True`.

This means that the relative paths used later (`initial_data/...`, `week_1/...`) should work correctly.

This check is useful because many `FileNotFoundError` issues in notebooks simply come from running the notebook from the wrong directory.

In [2]:
import ast
import re

# Read cumulative feedback containing rounds 1 and 2
inputs_text = Path("week_2/inputs.txt").read_text(encoding="utf-8-sig")
outputs_text = Path("week_2/outputs.txt").read_text(encoding="utf-8-sig")

# Convert NumPy array representations into plain lists
inputs_text = re.sub(
    r"array\((\[[^\[\]]*\])\)",
    r"\1",
    inputs_text
)

# Read each round safely, including lists spanning multiple lines
def read_rounds(text):
    tree = ast.parse(text.strip(), mode="exec")
    rounds = []

    for node in tree.body:
        if not isinstance(node, ast.Expr):
            raise ValueError("Unexpected format in the feedback file.")
        rounds.append(ast.literal_eval(node.value))

    return rounds


input_rounds = read_rounds(inputs_text)
output_rounds = read_rounds(outputs_text)

# Check that both files contain two rounds and eight functions per round
assert len(input_rounds) == len(output_rounds) == 2, (
    "Expected exactly two rounds in each cumulative file."
)

for round_inputs, round_outputs in zip(input_rounds, output_rounds):
    assert len(round_inputs) == len(round_outputs) == 8, (
        "Each round must contain results for eight functions."
    )

# Rebuild the updated data from the initial files every time
# This prevents duplicated observations when rerunning this cell
updated_data = {}

for function_number in range(1, 9):
    folder = Path("initial_data") / f"function_{function_number}"

    initial_X = np.load(folder / "initial_inputs.npy")
    initial_y = np.load(folder / "initial_outputs.npy").reshape(-1)

    new_X = np.asarray(
        [round_inputs[function_number - 1] for round_inputs in input_rounds],
        dtype=float
    )
    new_y = np.asarray(
        [round_outputs[function_number - 1] for round_outputs in output_rounds],
        dtype=float
    )

    assert initial_X.shape[0] == initial_y.size
    assert new_X.shape == (2, initial_X.shape[1])

    X = np.vstack([initial_X, new_X])
    y = np.concatenate([initial_y, new_y])

    assert np.isfinite(X).all() and np.isfinite(y).all()

    updated_data[function_number] = {"X": X, "y": y}

    print(
        f"Function {function_number}: "
        f"{len(initial_y)} → {len(y)} observations | "
        f"{X.shape[1]} variables"
    )

Function 1: 10 → 12 observations | 2 variables
Function 2: 10 → 12 observations | 2 variables
Function 3: 15 → 17 observations | 3 variables
Function 4: 30 → 32 observations | 4 variables
Function 5: 20 → 22 observations | 4 variables
Function 6: 20 → 22 observations | 5 variables
Function 7: 30 → 32 observations | 6 variables
Function 8: 40 → 42 observations | 8 variables


This cell reconstructs the dataset that should be used for the third optimisation round.

The initial observations are loaded first, and then the two previous submitted rounds are appended. Rebuilding the data from scratch is safer than modifying an already-updated object because rerunning the cell cannot accidentally duplicate observations.

The code also validates that:

- exactly two previous rounds are present;
- all eight functions are present in every round;
- each submitted point has the expected number of dimensions;
- all coordinates and returned outputs are finite.

The output confirms that each function now contains its initial dataset plus **two additional observations**:

- Function 1: `10 → 12`
- Function 2: `10 → 12`
- Function 3: `15 → 17`
- Function 4: `30 → 32`
- Function 5: `20 → 22`
- Function 6: `20 → 22`
- Function 7: `30 → 32`
- Function 8: `40 → 42`

Therefore, `updated_data` contains the complete information available before choosing the Round 3 queries.


In [3]:
import pandas as pd

# Compare each round against the best output available before it
results = []

for function_number in range(1, 9):
    y = updated_data[function_number]["y"]

    # The last two observations correspond to rounds 1 and 2
    initial_best = float(np.max(y[:-2]))
    round_1_output = float(y[-2])
    round_2_output = float(y[-1])

    best_after_round_1 = max(initial_best, round_1_output)
    current_best = max(best_after_round_1, round_2_output)

    results.append({
        "Function": function_number,
        "Initial best": initial_best,
        "Round 1 output": round_1_output,
        "Round 1 improved": round_1_output > initial_best,
        "Round 2 output": round_2_output,
        "Round 2 improved": round_2_output > best_after_round_1,
        "Current best": current_best
    })

results_summary = pd.DataFrame(results).set_index("Function")

# Scientific notation preserves very small values, especially for Function 1
display(
    results_summary.style.format({
        "Initial best": "{:.6e}",
        "Round 1 output": "{:.6e}",
        "Round 2 output": "{:.6e}",
        "Current best": "{:.6e}"
    })
)

print(
    "Functions improved in round 1:",
    int(results_summary["Round 1 improved"].sum())
)

print(
    "Functions improved in round 2:",
    int(results_summary["Round 2 improved"].sum())
)

,Initial best,Round 1 output,Round 1 improved,Round 2 output,Round 2 improved,Current best
Function,,,,,,
1,7.710875e-16,-2.999782e-60,False,-2.685947e-06,False,7.710875e-16
2,6.112052e-01,6.582395e-01,True,-1.589273e-03,False,6.582395e-01
3,-3.483531e-02,-1.847707e-01,False,-4.629653e-02,False,-3.483531e-02
4,-4.025542e+00,1.948721e-01,True,-1.004889e+00,False,1.948721e-01
5,1.088860e+03,2.612732e+03,True,1.618706e+03,False,2.612732e+03
6,-7.142649e-01,-1.038314e+00,False,-6.174861e-01,True,-6.174861e-01
7,1.364968e+00,2.097864e+00,True,4.467338e-01,False,2.097864e+00
8,9.598482e+00,9.802623e+00,True,9.820152e+00,True,9.820152e+00


Functions improved in round 1: 5
Functions improved in round 2: 2


This cell evaluates whether each previously submitted query improved on the **best value that was available before that round**.

Because the black-box objective is maximisation, a round is counted as an improvement only when the returned output is larger than the previous best.

The summary shows that:

- **Round 1 improved 5 functions**.
- **Round 2 improved 2 functions**.

This comparison is useful because it tells us that the previous strategy was more successful for some functions than for others. It also establishes the current incumbent value for each black box, which is the value that future queries should ideally exceed.


In [4]:
# Identify which functions improved in each round
for round_number in [1, 2]:
    improved = results_summary.index[
        results_summary[f"Round {round_number} improved"]
    ].tolist()

    print(f"Functions improved in round {round_number}: {improved}")

# Measure the improvement achieved in round 2
print("\nRound 2 improvements:")

for function_number in results_summary.index:
    row = results_summary.loc[function_number]

    if row["Round 2 improved"]:
        previous_best = max(
            row["Initial best"],
            row["Round 1 output"]
        )

        improvement = row["Round 2 output"] - previous_best

        print(
            f"Function {function_number}: "
            f"{previous_best:.6e} → {row['Current best']:.6e} | "
            f"Improvement: {improvement:.6e}"
        )

Functions improved in round 1: [2, 4, 5, 7, 8]
Functions improved in round 2: [6, 8]

Round 2 improvements:
Function 6: -7.142649e-01 → -6.174861e-01 | Improvement: 9.677886e-02
Function 8: 9.802623e+00 → 9.820152e+00 | Improvement: 1.752849e-02


This cell lists the functions that improved in each round and then measures the numerical improvement achieved in Round 2.

The output shows:

- Round 1 improvements: **Functions 2, 4, 5, 7 and 8**
- Round 2 improvements: **Functions 6 and 8**

For Round 2:

- Function 6 improved from approximately `−0.714265` to `−0.617486`, a gain of about `0.096779`.
- Function 8 improved from approximately `9.802623` to `9.820152`, a gain of about `0.017528`.

This gives context for the third iteration. It shows which functions are still failing to beat their existing best values and where a more exploratory strategy may be useful.


In [5]:
# Prepare binary labels for the SVM experiment
svm_data = {}
class_summary = []

for function_number in range(1, 9):
    X = updated_data[function_number]["X"]
    y = updated_data[function_number]["y"]

    # Define a fixed threshold using only the initial observations
    initial_y = y[:-2]
    threshold = float(np.median(initial_y))

    # Class 1: above the threshold
    # Class 0: at or below the threshold
    labels = (y > threshold).astype(int)

    # Keep the original continuous outputs unchanged
    svm_data[function_number] = {
        "X": X.copy(),
        "labels": labels,
        "threshold": threshold
    }

    class_summary.append({
        "Function": function_number,
        "Threshold": threshold,
        "Class 0 count": int(np.sum(labels == 0)),
        "Class 1 count": int(np.sum(labels == 1))
    })

class_summary = pd.DataFrame(class_summary).set_index("Function")

display(
    class_summary.style.format({
        "Threshold": "{:.6e}"
    })
)

,Threshold,Class 0 count,Class 1 count
Function,,,
1,6.793724e-80,7,5
2,2.297919e-01,6,6
3,-1.059650e-01,9,8
4,-1.604008e+01,15,17
5,6.394843e+01,10,12
6,-1.446370e+00,10,12
7,7.863144e-02,15,17
8,7.889150e+00,20,22


The original task is still a **continuous optimisation problem**, but this cell creates a binary classification view so that SVMs can be used as an additional diagnostic.

For each function, the classification threshold is fixed using the **median of the initial observations only**. The two Round 1 and Round 2 observations are deliberately excluded when defining the threshold, which avoids letting the threshold shift after seeing the later feedback.

The labels are:

- `1` = output above the initial-data median;
- `0` = output at or below that median.

The class counts are reasonably balanced across the eight functions. For example, Function 2 has 6 observations in each class and Function 8 has 20 vs 22.

This SVM classification does **not replace the original continuous outputs**. The Gaussian Process will still model the real `y` values. The SVM is only used later as an interpretability check to see whether a GP-UCB proposal lies in a region classified as above or below the chosen threshold.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_validate

# Compare two fixed SVM configurations before tuning hyperparameters
svm_models = {}
svm_results = []

for function_number in range(1, 9):
    X = svm_data[function_number]["X"]
    labels = svm_data[function_number]["labels"]

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    svm_models[function_number] = {}

    for kernel_name in ["linear", "rbf"]:
        # Fit scaling within each training fold to avoid data leakage
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(
                kernel=kernel_name,
                C=1.0,
                gamma="scale"
            ))
        ])

        scores = cross_validate(
            model,
            X,
            labels,
            cv=cv,
            scoring="balanced_accuracy",
            return_train_score=True,
            error_score="raise"
        )

        # Fit on all available observations for later candidate assessment
        model.fit(X, labels)
        svm_models[function_number][kernel_name] = model

        svm_results.append({
            "Function": function_number,
            "Kernel": kernel_name,
            "Training balanced accuracy": scores["train_score"].mean(),
            "CV balanced accuracy": scores["test_score"].mean(),
            "CV standard deviation": scores["test_score"].std(),
            "Support vectors": int(
                model.named_steps["svc"].n_support_.sum()
            )
        })

svm_results = pd.DataFrame(svm_results)

display(
    svm_results.style.format({
        "Training balanced accuracy": "{:.3f}",
        "CV balanced accuracy": "{:.3f}",
        "CV standard deviation": "{:.3f}"
    })
)

,Function,Kernel,Training balanced accuracy,CV balanced accuracy,CV standard deviation,Support vectors
0,1,linear,0.925,0.778,0.039,6
1,1,rbf,0.925,0.778,0.039,10
2,2,linear,0.792,0.500,0.000,10
3,2,rbf,0.792,0.750,0.000,11
4,3,linear,0.767,0.583,0.180,13
5,3,rbf,0.906,0.417,0.204,17
6,4,linear,0.717,0.722,0.126,21
7,4,rbf,0.936,0.683,0.095,24
8,5,linear,0.872,0.694,0.153,14
9,5,rbf,0.938,0.708,0.156,20


This cell compares a linear SVM and an RBF SVM using the same fixed value `C = 1`.

Each model is placed inside a `Pipeline` with `StandardScaler`. This is important because SVMs are sensitive to feature scale, and putting the scaler inside the pipeline ensures that scaling is learned separately within each cross-validation fold rather than leaking information from the validation fold.

The metric used is **balanced accuracy**, which is appropriate because some functions have slightly unequal class counts.

Some useful patterns in the output are:

- **Function 1:** linear and RBF both obtain CV balanced accuracy ≈ `0.778`.
- **Function 2:** RBF (`0.750`) clearly outperforms linear (`0.500`).
- **Function 3:** linear gives CV ≈ `0.583`, while RBF falls to ≈ `0.417` despite much higher training accuracy. This is a sign that the more flexible RBF model may be overfitting.
- **Function 7:** linear performs very strongly (`0.844`) while RBF drops to `0.678`.
- **Function 8:** both kernels perform well, with linear slightly higher (`0.839` vs `0.815`).

The comparison between training and cross-validation performance is especially informative: a very high training score combined with a weaker CV score suggests that the model is fitting the small observed dataset too closely.


In [7]:
from sklearn.model_selection import GridSearchCV

# Tune C while comparing linear and RBF kernels
svm_searches = {}
selected_svm_models = {}
tuning_results = []

for function_number in range(1, 9):
    X = svm_data[function_number]["X"]
    labels = svm_data[function_number]["labels"]

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(gamma="scale"))
    ])

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    # List simpler configurations first to resolve exact ties
    parameter_grid = [
        {
            "svc__kernel": ["linear"],
            "svc__C": [0.1, 1.0, 10.0]
        },
        {
            "svc__kernel": ["rbf"],
            "svc__C": [0.1, 1.0, 10.0]
        }
    ]

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grid,
        scoring="balanced_accuracy",
        cv=cv,
        refit=True,
        return_train_score=True,
        error_score="raise"
    )

    search.fit(X, labels)

    # GridSearchCV refits the selected model on all observations
    svm_searches[function_number] = search
    selected_svm_models[function_number] = search.best_estimator_

    best_index = search.best_index_

    tuning_results.append({
        "Function": function_number,
        "Selected kernel": search.best_params_["svc__kernel"],
        "Selected C": search.best_params_["svc__C"],
        "Training balanced accuracy":
            search.cv_results_["mean_train_score"][best_index],
        "Selection CV score": search.best_score_,
        "CV standard deviation":
            search.cv_results_["std_test_score"][best_index]
    })

tuning_summary = pd.DataFrame(tuning_results).set_index("Function")

display(
    tuning_summary.style.format({
        "Selected C": "{:.1f}",
        "Training balanced accuracy": "{:.3f}",
        "Selection CV score": "{:.3f}",
        "CV standard deviation": "{:.3f}"
    })
)

,Selected kernel,Selected C,Training balanced accuracy,Selection CV score,CV standard deviation
Function,,,,,
1,linear,1.0,0.925,0.778,0.039
2,rbf,0.1,0.792,0.750,0.000
3,linear,1.0,0.767,0.583,0.180
4,rbf,10.0,1.000,0.778,0.087
5,rbf,10.0,1.000,0.875,0.177
6,rbf,10.0,1.000,0.806,0.142
7,linear,1.0,0.799,0.844,0.042
8,linear,0.1,0.931,0.863,0.008


This cell performs a controlled grid search over:

- kernel: `linear` or `rbf`;
- `C`: `0.1`, `1.0`, or `10.0`.

The same stratified 3-fold cross-validation and balanced-accuracy metric are used. The best model for each function is then refitted on all available observations and stored in `selected_svm_models`.

The selected configurations are:

- F1: **linear, C = 1**
- F2: **RBF, C = 0.1**
- F3: **linear, C = 1**
- F4: **RBF, C = 10**
- F5: **RBF, C = 10**
- F6: **RBF, C = 10**
- F7: **linear, C = 1**
- F8: **linear, C = 0.1**

The strongest selection CV scores are obtained for F5, F7 and F8. F3 remains the most difficult classification problem.

The SVM is still **not used to choose the optimisation point directly**. It will only classify the GP-UCB proposal as above or below the performance threshold. This keeps the Gaussian Process and UCB as the main optimisation method.


In [8]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

# Use all Function 1 observations available for round 3
X_1 = updated_data[1]["X"]
y_1 = updated_data[1]["y"]

models_1 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_1.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_1, y_1)
    models_1[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_1))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Observations: 12
Fitted kernel: 1.02**2 * Matern(length_scale=[2, 0.0267], nu=2.5) + WhiteKernel(noise_level=2.87e-07)
Log-marginal likelihood: -16.724814446989278

Model: B_wider
Observations: 12
Fitted kernel: 1.02**2 * Matern(length_scale=[10, 0.0269], nu=2.5) + WhiteKernel(noise_level=2.53e-08)
Log-marginal likelihood: -16.717250422356756


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Two Gaussian Processes are fitted with the same Matérn-5/2 structure but different upper bounds for the length scales:

- `A_original`: upper bound = `2`
- `B_wider`: upper bound = `10`

The results are:

- A_original LML = `−16.724814`
- B_wider LML = `−16.717250`

The first length scale reaches the upper bound in both models (`2` and then `10`). This suggests that the GP would prefer Function 1 to be even smoother in that dimension.

However, widening the bound improves the log-marginal likelihood by only about **0.0076**, which is negligible. The two fits are therefore effectively equivalent.

The white-noise term is also extremely small, meaning that the GP treats the observations as nearly deterministic.

For this function, there is no strong statistical reason to prefer the wider model.


In [9]:
# Generate the same two-dimensional candidates for both GP models
rng_1 = np.random.default_rng(42)
candidates_1 = rng_1.uniform(
    0, 0.999999, size=(100000, X_1.shape[1])
)

# Round to portal-ready coordinates
candidates_1 = np.unique(np.round(candidates_1, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_1 = np.min(
    np.linalg.norm(
        candidates_1[:, None, :] - X_1[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_1 > 1e-6
candidates_1 = candidates_1[keep]
nearest_distance_1 = nearest_distance_1[keep]

# Compare UCB proposals without filtering candidates by SVM class
beta_1 = 2.0
proposals_1 = {}

for name, model in models_1.items():
    mean, std = model.predict(candidates_1, return_std=True)
    ucb = mean + beta_1 * std

    index = np.argmax(ucb)
    query = candidates_1[index]

    svm_class = int(
        selected_svm_models[1].predict(query.reshape(1, -1))[0]
    )

    proposals_1[name] = {
        "query": query.copy(),
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index]),
        "nearest_distance": float(nearest_distance_1[index]),
        "svm_class": svm_class
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance_1[index]:.6f}"
    )
    print("SVM class (1 = above threshold):", svm_class)


Model: A_original
Proposed query: 0.881159-0.530910
Predicted output: -2.502833e-04
Predictive standard deviation: 9.996244e-04
UCB score: 1.748965e-03
Distance to nearest observation: 0.051417
SVM class (1 = above threshold): 1

Model: B_wider
Proposed query: 0.841418-0.530599
Predicted output: -2.499252e-04
Predictive standard deviation: 1.000211e-03
UCB score: 1.750496e-03
Distance to nearest observation: 0.066874
SVM class (1 = above threshold): 1


### 💬 Function 1 — UCB proposals and SVM diagnostic

Both Gaussian Processes evaluate the **same 100,000 candidate points**, so differences in the proposed query are due to the model rather than different random samples.

The acquisition function is:

**UCB = predicted mean + 2 × predictive standard deviation**

This balances:

- **exploitation** through the predicted mean;
- **exploration** through predictive uncertainty.

Results:

**A_original**
- query: `0.881159-0.530910`
- mean ≈ `−0.0002503`
- std ≈ `0.0009996`
- UCB ≈ `0.0017490`
- nearest distance ≈ `0.0514`
- SVM class = `1`

**B_wider**
- query: `0.841418-0.530599`
- mean ≈ `−0.0002499`
- std ≈ `0.0010002`
- UCB ≈ `0.0017505`
- nearest distance ≈ `0.0669`
- SVM class = `1`

The two proposals are almost indistinguishable. B_wider has a trivially larger UCB, but its GP fit was not meaningfully better.

Both proposals are classified by the SVM as above the performance threshold. The small nearest-neighbour distances show that this is a relatively **exploitative** search around already sampled regions.


In [11]:

# Store the selected model and query for Function 1, round 3
selected_model_1 = models_1["A_original"]
query_1_round3 = proposals_1["A_original"]["query"].copy()

# Keep the prediction details for comparison with future feedback
selected_proposal_1_round3 = proposals_1["A_original"].copy()

print(
    "Function 1 — round 3:",
    "-".join(f"{x:.6f}" for x in query_1_round3)
)

Function 1 — round 3: 0.881159-0.530910


### Function 1 — final Round 3 decision

The selected model is **A_original**, giving:

`0.881159-0.530910`

This is a conservative choice. Since widening the bounds produced almost no improvement in LML and only a negligible difference in UCB, there is no strong reason to introduce the wider fit.

The full proposal information is stored as well so that, once the true black-box output is returned, the prediction and uncertainty can be compared with the observed result.


In [12]:
# Use all Function 2 observations available for round 3
X_2 = updated_data[2]["X"]
y_2 = updated_data[2]["y"]

models_2 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_2.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_2, y_2)
    models_2[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_2))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Observations: 12
Fitted kernel: 0.912**2 * Matern(length_scale=[0.0557, 2], nu=2.5) + WhiteKernel(noise_level=0.00953)
Log-marginal likelihood: -12.287773690801924

Model: B_wider
Observations: 12
Fitted kernel: 0.965**2 * Matern(length_scale=[0.0636, 10], nu=2.5) + WhiteKernel(noise_level=0.014)
Log-marginal likelihood: -11.774219429199201


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Function 2 — Gaussian Process model comparison

For Function 2, the second length scale reaches the upper bound in both fits.

Results:

- A_original: `Matern([0.0557, 2])`, LML = `−12.287774`
- B_wider: `Matern([0.0636, 10])`, LML = `−11.774219`

The improvement is approximately:

**ΔLML ≈ +0.514**

This is much more meaningful than the tiny difference observed for Function 1.

The first length scale remains very small, which means the model expects the function to change relatively quickly along the first dimension. The second length scale is extremely large, meaning much smoother behaviour along the second dimension.

Because the wider model improves the fit while allowing the second dimension to become smoother, **B_wider is better supported by the data**.


In [13]:
# Generate the same two-dimensional candidates for both GP models
rng_2 = np.random.default_rng(42)
candidates_2 = rng_2.uniform(
    0, 0.999999, size=(100000, X_2.shape[1])
)

# Round to portal-ready coordinates
candidates_2 = np.unique(np.round(candidates_2, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_2 = np.min(
    np.linalg.norm(
        candidates_2[:, None, :] - X_2[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_2 > 1e-6
candidates_2 = candidates_2[keep]
nearest_distance_2 = nearest_distance_2[keep]

# Compare UCB proposals without filtering candidates by SVM class
beta_2 = 2.0
proposals_2 = {}

for name, model in models_2.items():
    mean, std = model.predict(candidates_2, return_std=True)
    ucb = mean + beta_2 * std

    index = np.argmax(ucb)
    query = candidates_2[index]

    svm_class = int(
        selected_svm_models[2].predict(query.reshape(1, -1))[0]
    )

    proposals_2[name] = {
        "query": query.copy(),
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index]),
        "nearest_distance": float(nearest_distance_2[index]),
        "svm_class": svm_class
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance_2[index]:.6f}"
    )
    print("SVM class (1 = above threshold):", svm_class)


Model: A_original
Proposed query: 0.746052-0.994981
Predicted output: 5.158722e-01
Predictive standard deviation: 1.451905e-01
UCB score: 8.062533e-01
Distance to nearest observation: 0.081029
SVM class (1 = above threshold): 1

Model: B_wider
Proposed query: 0.752835-0.997401
Predicted output: 5.539196e-01
Predictive standard deviation: 1.214022e-01
UCB score: 7.967239e-01
Distance to nearest observation: 0.086820
SVM class (1 = above threshold): 1


### Function 2 — UCB comparison

The proposals are:

**A_original**
- query: `0.746052-0.994981`
- mean ≈ `0.515872`
- std ≈ `0.145191`
- UCB ≈ `0.806253`
- nearest distance ≈ `0.0810`
- SVM class = `1`

**B_wider**
- query: `0.752835-0.997401`
- mean ≈ `0.553920`
- std ≈ `0.121402`
- UCB ≈ `0.796724`
- nearest distance ≈ `0.0868`
- SVM class = `1`

A_original has the slightly higher UCB because its uncertainty is larger. B_wider, however, has the **higher predicted mean** and a substantially better log-marginal likelihood.

This is an important distinction: the largest UCB does not automatically imply the best underlying GP fit. Here we prefer the model that explains the observed data better, while still proposing a strong point in a region that the SVM classifies as above threshold.


In [14]:
# Store the selected model and query for Function 2, round 3
selected_model_2 = models_2["B_wider"]
query_2_round3 = proposals_2["B_wider"]["query"].copy()

# Keep the prediction details for comparison with future feedback
selected_proposal_2_round3 = proposals_2["B_wider"].copy()

print(
    "Function 2 — round 3:",
    "-".join(f"{x:.6f}" for x in query_2_round3)
)

Function 2 — round 3: 0.752835-0.997401


### Function 2 — final Round 3 decision

The selected model is **B_wider**, and the query stored for Round 3 is:

`0.752835-0.997401`

The choice is driven mainly by the improved GP fit and the higher predicted mean, rather than by the small UCB difference in favour of A_original.

This is a good example of using the acquisition function together with model diagnostics rather than following UCB mechanically.

In [15]:
# Use all Function 3 observations available for round 3
X_3 = updated_data[3]["X"]
y_3 = updated_data[3]["y"]

models_3 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_3.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_3, y_3)
    models_3[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_3))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Observations: 17
Fitted kernel: 1.14**2 * Matern(length_scale=[2, 2, 0.0618], nu=2.5) + WhiteKernel(noise_level=0.00929)
Log-marginal likelihood: -19.137749124297716

Model: B_wider
Observations: 17
Fitted kernel: 1.34**2 * Matern(length_scale=[10, 10, 0.0796], nu=2.5) + WhiteKernel(noise_level=0.0157)
Log-marginal likelihood: -17.68480610820923


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Function 3 — both first dimensions prefer very large length scales

Results:

- A_original: `Matern([2, 2, 0.0618])`, LML = `−19.137749`
- B_wider: `Matern([10, 10, 0.0796])`, LML = `−17.684806`

The first two dimensions hit the upper bound in both models. When the bound is widened from 2 to 10, the log-marginal likelihood improves by about:

**ΔLML ≈ +1.453**

That is a meaningful improvement.

The fitted length scales reveal a strong contrast:

- x1 and x2 have very large length scales, so the model sees relatively smooth behaviour along those dimensions;
- x3 has a very small length scale (`~0.06–0.08`), so the output may change much more rapidly with respect to the third variable.

The wider model therefore has a clearly better-supported fit.


In [16]:
# Generate the same three-dimensional candidates for both GP models
rng_3 = np.random.default_rng(42)
candidates_3 = rng_3.uniform(
    0, 0.999999, size=(100000, X_3.shape[1])
)

# Round to portal-ready coordinates
candidates_3 = np.unique(np.round(candidates_3, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_3 = np.min(
    np.linalg.norm(
        candidates_3[:, None, :] - X_3[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_3 > 1e-6
candidates_3 = candidates_3[keep]
nearest_distance_3 = nearest_distance_3[keep]

# Compare UCB proposals without filtering candidates by SVM class
beta_3 = 2.0
proposals_3 = {}

for name, model in models_3.items():
    mean, std = model.predict(candidates_3, return_std=True)
    ucb = mean + beta_3 * std

    index = np.argmax(ucb)
    query = candidates_3[index]

    svm_class = int(
        selected_svm_models[3].predict(query.reshape(1, -1))[0]
    )

    proposals_3[name] = {
        "query": query.copy(),
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index]),
        "nearest_distance": float(nearest_distance_3[index]),
        "svm_class": svm_class
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance_3[index]:.6f}"
    )
    print("SVM class (1 = above threshold):", svm_class)
    


Model: A_original
Proposed query: 0.002920-0.004178-0.641890
Predicted output: -9.434017e-02
Predictive standard deviation: 8.862037e-02
UCB score: 8.290056e-02
Distance to nearest observation: 0.264781
SVM class (1 = above threshold): 0

Model: B_wider
Proposed query: 0.050178-0.066268-0.655743
Predicted output: -9.603975e-02
Predictive standard deviation: 9.390945e-02
UCB score: 9.177915e-02
Distance to nearest observation: 0.201154
SVM class (1 = above threshold): 0


### Function 3 — UCB and the exploration/exploitation balance

Results:

**A_original**
- query: `0.002920-0.004178-0.641890`
- mean ≈ `−0.09434`
- std ≈ `0.08862`
- UCB ≈ `0.08290`
- nearest distance ≈ `0.2648`
- SVM class = `0`

**B_wider**
- query: `0.050178-0.066268-0.655743`
- mean ≈ `−0.09604`
- std ≈ `0.09391`
- UCB ≈ `0.09178`
- nearest distance ≈ `0.2012`
- SVM class = `0`

B_wider gives the higher UCB because its predictive uncertainty is slightly larger, even though its predicted mean is slightly lower.

Both candidate points are classified by the SVM as **below the initial median threshold**. This does not automatically invalidate them: the SVM is only a diagnostic, while the GP-UCB calculation works with the full continuous response and uncertainty.

Because B_wider has both the better LML and the higher UCB, it is the stronger choice here.


In [17]:
# Store the selected model and query for Function 3, round 3
selected_model_3 = models_3["B_wider"]
query_3_round3 = proposals_3["B_wider"]["query"].copy()

# Keep the prediction details for comparison with future feedback
selected_proposal_3_round3 = proposals_3["B_wider"].copy()

print(
    "Function 3 — round 3:",
    "-".join(f"{x:.6f}" for x in query_3_round3)
)

Function 3 — round 3: 0.050178-0.066268-0.655743


### Function 3 — final Round 3 decision

The selected model is **B_wider** and the stored query is:

`0.050178-0.066268-0.655743`

The SVM classification is `0`, but we do not reject the point for that reason. The continuous Gaussian Process contains more information than a binary above/below-median label, and UCB deliberately values uncertainty as well as predicted performance.


In [18]:
# Use all Function 4 observations available for round 3
X_4 = updated_data[4]["X"]
y_4 = updated_data[4]["y"]

models_4 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_4.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_4, y_4)
    models_4[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_4))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )


Model: A_original
Observations: 32
Fitted kernel: 2.79**2 * Matern(length_scale=[1.53, 1.38, 1.41, 1.39], nu=2.5) + WhiteKernel(noise_level=0.000647)
Log-marginal likelihood: -16.03631486701597

Model: B_wider
Observations: 32
Fitted kernel: 2.79**2 * Matern(length_scale=[1.53, 1.38, 1.41, 1.39], nu=2.5) + WhiteKernel(noise_level=0.000647)
Log-marginal likelihood: -16.036314867023442


### Function 4 — the two GP fits are effectively identical

The fitted models are:

- A_original LML = `−16.0363148670`
- B_wider LML = `−16.0363148670`

Their kernels are also identical:

`Matern(length_scale=[1.53, 1.38, 1.41, 1.39], nu=2.5)`

None of the length scales needs the wider bound, so increasing the allowed maximum from 2 to 10 has no practical effect.

The white-noise estimate is small (`~0.000647`), meaning that most of the observed variation can be explained by the smooth GP component.

Since the wider model adds no benefit, **A_original is retained**.


In [19]:
# Keep the original bounds: both fits are practically identical
selected_model_4 = models_4["A_original"]

# Generate four-dimensional candidate points
rng_4 = np.random.default_rng(42)
candidates_4 = rng_4.uniform(
    0, 0.999999, size=(100000, X_4.shape[1])
)

# Round to portal-ready coordinates
candidates_4 = np.unique(np.round(candidates_4, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_4 = np.min(
    np.linalg.norm(
        candidates_4[:, None, :] - X_4[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_4 > 1e-6
candidates_4 = candidates_4[keep]
nearest_distance_4 = nearest_distance_4[keep]

# Select the candidate with the highest UCB
beta_4 = 2.0
mean, std = selected_model_4.predict(
    candidates_4, return_std=True
)
ucb = mean + beta_4 * std

index = np.argmax(ucb)
query_4_round3 = candidates_4[index].copy()

# Record the SVM classification without filtering candidates
svm_class = int(
    selected_svm_models[4].predict(
        query_4_round3.reshape(1, -1)
    )[0]
)

# Store prediction details for comparison with future feedback
selected_proposal_4_round3 = {
    "query": query_4_round3.copy(),
    "mean": float(mean[index]),
    "std": float(std[index]),
    "ucb": float(ucb[index]),
    "nearest_distance": float(nearest_distance_4[index]),
    "svm_class": svm_class
}

print(
    "Function 4 — round 3:",
    "-".join(f"{x:.6f}" for x in query_4_round3)
)
print(f"Predicted output: {mean[index]:.6e}")
print(f"Predictive standard deviation: {std[index]:.6e}")
print(f"UCB score: {ucb[index]:.6e}")
print(
    f"Distance to nearest observation: "
    f"{nearest_distance_4[index]:.6f}"
)
print("SVM class (1 = above threshold):", svm_class)

Function 4 — round 3: 0.368862-0.383152-0.454070-0.450029
Predicted output: -1.633536e-01
Predictive standard deviation: 5.466365e-01
UCB score: 9.299195e-01
Distance to nearest observation: 0.110482
SVM class (1 = above threshold): 1


### Function 4 — direct Round 3 proposal

Because the two fitted models are effectively identical, the notebook keeps A_original and directly searches for the best UCB candidate.

The selected query is:

`0.368862-0.383152-0.454070-0.450029`

with:

- predicted mean ≈ `−0.163354`
- std ≈ `0.546637`
- UCB ≈ `0.929920`
- nearest distance ≈ `0.1105`
- SVM class = `1`

The predicted mean itself is negative, but the uncertainty is large enough that the UCB becomes strongly positive. This is exactly how UCB encourages exploration of points that might perform much better than their current mean prediction suggests.

The relatively small nearest-observation distance means the point is still fairly close to known data, so the search is not purely exploratory.


In [20]:
# Use all Function 5 observations available for round 3
X_5 = updated_data[5]["X"]
y_5 = updated_data[5]["y"]

models_5 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_5.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_5, y_5)
    models_5[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_5))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Observations: 22
Fitted kernel: 1.13**2 * Matern(length_scale=[2, 1.26, 0.805, 0.22], nu=2.5) + WhiteKernel(noise_level=7.52e-10)
Log-marginal likelihood: -17.766958874910344

Model: B_wider
Observations: 22
Fitted kernel: 1.62**2 * Matern(length_scale=[10, 1.65, 1.12, 0.326], nu=2.5) + WhiteKernel(noise_level=1.18e-09)
Log-marginal likelihood: -15.360902015418628


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Function 5 — widening the first length-scale bound improves the fit

Results:

- A_original: `Matern([2, 1.26, 0.805, 0.22])`, LML = `−17.766959`
- B_wider: `Matern([10, 1.65, 1.12, 0.326])`, LML = `−15.360902`

The first dimension hits the upper bound in both models. Widening the bound produces an LML improvement of approximately:

**ΔLML ≈ +2.406**

This is substantial, so the original bound was clearly restricting the GP.

The fourth dimension has the smallest length scale, meaning that the response is modelled as changing more locally along that variable. The first dimension, by contrast, appears very smooth.

The white-noise terms are essentially zero, so the function is treated as almost deterministic.


In [21]:
# Generate the same four-dimensional candidates for both GP models
rng_5 = np.random.default_rng(42)
candidates_5 = rng_5.uniform(
    0, 0.999999, size=(100000, X_5.shape[1])
)

# Round to portal-ready coordinates
candidates_5 = np.unique(np.round(candidates_5, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_5 = np.min(
    np.linalg.norm(
        candidates_5[:, None, :] - X_5[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_5 > 1e-6
candidates_5 = candidates_5[keep]
nearest_distance_5 = nearest_distance_5[keep]

# Compare UCB proposals without filtering candidates by SVM class
beta_5 = 2.0
proposals_5 = {}

for name, model in models_5.items():
    mean, std = model.predict(candidates_5, return_std=True)
    ucb = mean + beta_5 * std

    index = np.argmax(ucb)
    query = candidates_5[index]

    svm_class = int(
        selected_svm_models[5].predict(query.reshape(1, -1))[0]
    )

    proposals_5[name] = {
        "query": query.copy(),
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index]),
        "nearest_distance": float(nearest_distance_5[index]),
        "svm_class": svm_class
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance_5[index]:.6f}"
    )
    print("SVM class (1 = above threshold):", svm_class)


Model: A_original
Proposed query: 0.717512-0.908905-0.977761-0.998905
Predicted output: 2.547705e+03
Predictive standard deviation: 1.798609e+02
UCB score: 2.907427e+03
Distance to nearest observation: 0.428438
SVM class (1 = above threshold): 1

Model: B_wider
Proposed query: 0.505572-0.996280-0.970921-0.997435
Predicted output: 2.668558e+03
Predictive standard deviation: 9.047195e+01
UCB score: 2.849502e+03
Distance to nearest observation: 0.275101
SVM class (1 = above threshold): 1


### Function 5 — comparing the two UCB proposals

Results:

**A_original**
- query: `0.717512-0.908905-0.977761-0.998905`
- mean ≈ `2547.7`
- std ≈ `179.9`
- UCB ≈ `2907.4`
- nearest distance ≈ `0.4284`
- SVM class = `1`

**B_wider**
- query: `0.505572-0.996280-0.970921-0.997435`
- mean ≈ `2668.6`
- std ≈ `90.5`
- UCB ≈ `2849.5`
- nearest distance ≈ `0.2751`
- SVM class = `1`

A_original has the higher UCB, but this is largely because its uncertainty is almost twice as high.

B_wider has:

- a much better GP fit;
- a higher predicted mean;
- lower uncertainty;
- an SVM classification of `1`.

Therefore, the slightly lower UCB does not make it the weaker option. The wider GP gives a more credible and more confident prediction, so B_wider is selected.


In [22]:
# Store the selected model and query for Function 5, round 3
selected_model_5 = models_5["B_wider"]
query_5_round3 = proposals_5["B_wider"]["query"].copy()

# Keep the prediction details for comparison with future feedback
selected_proposal_5_round3 = proposals_5["B_wider"].copy()

print(
    "Function 5 — round 3:",
    "-".join(f"{x:.6f}" for x in query_5_round3)
)

Function 5 — round 3: 0.505572-0.996280-0.970921-0.997435


### Function 5 — final Round 3 decision

The selected query is:

`0.505572-0.996280-0.970921-0.997435`

from **B_wider**.

This choice demonstrates why model selection is not based only on the largest acquisition value. A_original has the larger UCB because it is much more uncertain, while B_wider fits the existing observations considerably better and predicts a higher mean response.

In [23]:
# Use all Function 6 observations available for round 3
X_6 = updated_data[6]["X"]
y_6 = updated_data[6]["y"]

models_6 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_6.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_6, y_6)
    models_6[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_6))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )


Model: A_original
Observations: 22
Fitted kernel: 1.4**2 * Matern(length_scale=[0.557, 1.16, 1.03, 1.11, 1.24], nu=2.5) + WhiteKernel(noise_level=0.0358)
Log-marginal likelihood: -24.146734398621938

Model: B_wider
Observations: 22
Fitted kernel: 1.4**2 * Matern(length_scale=[0.557, 1.16, 1.03, 1.11, 1.24], nu=2.5) + WhiteKernel(noise_level=0.0358)
Log-marginal likelihood: -24.146734399119122


### Function 6 — no evidence that wider bounds are needed

The two fitted models are essentially identical:

- A_original LML = `−24.1467343986`
- B_wider LML = `−24.1467343991`

Both produce:

`Matern([0.557, 1.16, 1.03, 1.11, 1.24]) + WhiteKernel(0.0358)`

No length scale is reaching the original upper bound of 2, so widening it to 10 does not help.

The non-negligible white-noise term (`0.0358`) is also notable. Unlike several other functions, the GP is allowing some residual variation that is not captured perfectly by the smooth Matérn component.

Because the fits are indistinguishable, **A_original is retained**.


In [24]:
# Keep the original bounds: both fits are practically identical
selected_model_6 = models_6["A_original"]

# Generate five-dimensional candidate points
rng_6 = np.random.default_rng(42)
candidates_6 = rng_6.uniform(
    0, 0.999999, size=(100000, X_6.shape[1])
)

# Round to portal-ready coordinates
candidates_6 = np.unique(np.round(candidates_6, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_6 = np.min(
    np.linalg.norm(
        candidates_6[:, None, :] - X_6[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_6 > 1e-6
candidates_6 = candidates_6[keep]
nearest_distance_6 = nearest_distance_6[keep]

# Select the candidate with the highest UCB
beta_6 = 2.0
mean, std = selected_model_6.predict(
    candidates_6, return_std=True
)
ucb = mean + beta_6 * std

index = np.argmax(ucb)
query_6_round3 = candidates_6[index].copy()

# Record the SVM classification without filtering candidates
svm_class = int(
    selected_svm_models[6].predict(
        query_6_round3.reshape(1, -1)
    )[0]
)

# Store prediction details for comparison with future feedback
selected_proposal_6_round3 = {
    "query": query_6_round3.copy(),
    "mean": float(mean[index]),
    "std": float(std[index]),
    "ucb": float(ucb[index]),
    "nearest_distance": float(nearest_distance_6[index]),
    "svm_class": svm_class
}

print(
    "Function 6 — round 3:",
    "-".join(f"{x:.6f}" for x in query_6_round3)
)
print(f"Predicted output: {mean[index]:.6e}")
print(f"Predictive standard deviation: {std[index]:.6e}")
print(f"UCB score: {ucb[index]:.6e}")
print(
    f"Distance to nearest observation: "
    f"{nearest_distance_6[index]:.6f}"
)
print("SVM class (1 = above threshold):", svm_class)


Function 6 — round 3: 0.546223-0.109744-0.472859-0.810424-0.001736
Predicted output: -5.867368e-01
Predictive standard deviation: 1.676386e-01
UCB score: -2.514595e-01
Distance to nearest observation: 0.321560
SVM class (1 = above threshold): 1


### Function 6 — Round 3 proposal

Using A_original, the selected query is:

`0.546223-0.109744-0.472859-0.810424-0.001736`

with:

- predicted mean ≈ `−0.586737`
- std ≈ `0.167639`
- UCB ≈ `−0.251460`
- nearest distance ≈ `0.3216`
- SVM class = `1`

The UCB is negative, but that is not a problem. UCB only needs to be **larger than the UCB values of the other candidate points**; there is no requirement that it be positive.

The SVM classifies the point as above the initial median threshold, which provides a useful secondary indication that the region may be promising.

The distance of about 0.32 shows a moderate amount of exploration in the five-dimensional domain.


In [25]:
# Use all Function 7 observations available for round 3
X_7 = updated_data[7]["X"]
y_7 = updated_data[7]["y"]

models_7 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_7.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_7, y_7)
    models_7[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_7))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Observations: 32
Fitted kernel: 0.907**2 * Matern(length_scale=[0.799, 0.591, 2, 2, 0.18, 0.115], nu=2.5) + WhiteKernel(noise_level=1.11e-09)
Log-marginal likelihood: -36.33416378317804

Model: B_wider
Observations: 32
Fitted kernel: 0.929**2 * Matern(length_scale=[0.829, 0.625, 10, 10, 0.175, 0.124], nu=2.5) + WhiteKernel(noise_level=4.47e-09)
Log-marginal likelihood: -35.78814319648781


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


### Function 7 — dimensions 3 and 4 prefer much larger length scales

Results:

- A_original: `Matern([0.799, 0.591, 2, 2, 0.18, 0.115])`, LML = `−36.334164`
- B_wider: `Matern([0.829, 0.625, 10, 10, 0.175, 0.124])`, LML = `−35.788143`

The third and fourth dimensions hit the upper bound in both fits. Widening the bound improves the LML by about:

**ΔLML ≈ +0.546**

This is a moderate but real improvement.

The very large length scales for x3 and x4 suggest that, over the observed region, the GP sees relatively slow variation along these dimensions. By contrast, x5 and x6 have very short length scales and therefore much more local variation.

The white-noise term is almost zero, so the observations are again treated as essentially deterministic.


In [26]:
# Generate the same six-dimensional candidates for both GP models
rng_7 = np.random.default_rng(42)
candidates_7 = rng_7.uniform(
    0, 0.999999, size=(100000, X_7.shape[1])
)

# Round to portal-ready coordinates
candidates_7 = np.unique(np.round(candidates_7, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_7 = np.min(
    np.linalg.norm(
        candidates_7[:, None, :] - X_7[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_7 > 1e-6
candidates_7 = candidates_7[keep]
nearest_distance_7 = nearest_distance_7[keep]

# Compare UCB proposals without filtering candidates by SVM class
beta_7 = 2.0
proposals_7 = {}

for name, model in models_7.items():
    mean, std = model.predict(candidates_7, return_std=True)
    ucb = mean + beta_7 * std

    index = np.argmax(ucb)
    query = candidates_7[index]

    svm_class = int(
        selected_svm_models[7].predict(query.reshape(1, -1))[0]
    )

    proposals_7[name] = {
        "query": query.copy(),
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index]),
        "nearest_distance": float(nearest_distance_7[index]),
        "svm_class": svm_class
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance_7[index]:.6f}"
    )
    print("SVM class (1 = above threshold):", svm_class)


Model: A_original
Proposed query: 0.025633-0.228828-0.504650-0.013419-0.411166-0.790687
Predicted output: 1.964759e+00
Predictive standard deviation: 1.576493e-01
UCB score: 2.280058e+00
Distance to nearest observation: 0.148231
SVM class (1 = above threshold): 1

Model: B_wider
Proposed query: 0.086618-0.151507-0.205694-0.787945-0.397953-0.790895
Predicted output: 1.976501e+00
Predictive standard deviation: 1.616568e-01
UCB score: 2.299814e+00
Distance to nearest observation: 0.494250
SVM class (1 = above threshold): 1


### Function 7 — UCB strongly favours the wider-model proposal

Results:

**A_original**
- query: `0.025633-0.228828-0.504650-0.013419-0.411166-0.790687`
- mean ≈ `1.964759`
- std ≈ `0.157649`
- UCB ≈ `2.280058`
- nearest distance ≈ `0.1482`
- SVM class = `1`

**B_wider**
- query: `0.086618-0.151507-0.205694-0.787945-0.397953-0.790895`
- mean ≈ `1.976501`
- std ≈ `0.161657`
- UCB ≈ `2.299814`
- nearest distance ≈ `0.4943`
- SVM class = `1`

B_wider has:

- a slightly higher predicted mean;
- slightly greater uncertainty;
- a higher UCB;
- a better GP fit.

The most striking difference is the nearest-observation distance. The B_wider proposal is much farther from existing data, so it represents substantially more **exploration**.

Since both the model diagnostics and the acquisition function support B_wider, the selection is straightforward.


In [27]:
# Store the selected model and query for Function 7, round 3
selected_model_7 = models_7["B_wider"]
query_7_round3 = proposals_7["B_wider"]["query"].copy()

# Keep the prediction details for comparison with future feedback
selected_proposal_7_round3 = proposals_7["B_wider"].copy()

print(
    "Function 7 — round 3:",
    "-".join(f"{x:.6f}" for x in query_7_round3)
)

Function 7 — round 3: 0.086618-0.151507-0.205694-0.787945-0.397953-0.790895


### Function 7 — final Round 3 decision

The selected query is:

`0.086618-0.151507-0.205694-0.787945-0.397953-0.790895`

using **B_wider**.

This point is particularly useful for the exploration side of the exploration/exploitation trade-off because it is far from the nearest existing observation while still having a strong predicted mean and UCB.

In [28]:
# Use all Function 8 observations available for round 3
X_8 = updated_data[8]["X"]
y_8 = updated_data[8]["y"]

models_8 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_8.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_8, y_8)
    models_8[name] = model

    print(f"\nModel: {name}")
    print("Observations:", len(y_8))
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-pack


Model: A_original
Observations: 42
Fitted kernel: 1.02**2 * Matern(length_scale=[1.18, 2, 0.914, 2, 2, 2, 1.25, 2], nu=2.5) + WhiteKernel(noise_level=1.04e-10)
Log-marginal likelihood: -15.201643836115046

Model: B_wider
Observations: 42
Fitted kernel: 3.23**2 * Matern(length_scale=[3.01, 4.27, 2.2, 7.98, 10, 4.39, 2.92, 10], nu=2.5) + WhiteKernel(noise_level=1e-10)
Log-marginal likelihood: -0.9186009202070551


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### Function 8 — the original length-scale bounds are strongly restrictive

Function 8 provides the clearest evidence that the original upper bound of 2 is too small.

Results:

- A_original LML = `−15.201644`
- B_wider LML = `−0.918601`

The improvement is enormous:

**ΔLML ≈ +14.283**

In A_original, several length scales are stuck at the upper bound of 2. After widening the range, the fitted values increase substantially:

`[3.01, 4.27, 2.2, 7.98, 10, 4.39, 2.92, 10]`

Even with the wider bound, dimensions 5 and 8 still reach 10, suggesting that the response may be even smoother in those directions.

The noise estimate is essentially at its lower bound, so the GP again treats the data as almost deterministic.

The huge LML improvement makes **B_wider clearly preferable** as the statistical model.


In [29]:
# Generate the same eight-dimensional candidates for both GP models
rng_8 = np.random.default_rng(42)
candidates_8 = rng_8.uniform(
    0, 0.999999, size=(100000, X_8.shape[1])
)

# Round to portal-ready coordinates
candidates_8 = np.unique(np.round(candidates_8, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_8 = np.min(
    np.linalg.norm(
        candidates_8[:, None, :] - X_8[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_8 > 1e-6
candidates_8 = candidates_8[keep]
nearest_distance_8 = nearest_distance_8[keep]

# Compare UCB proposals without filtering candidates by SVM class
beta_8 = 2.0
proposals_8 = {}

for name, model in models_8.items():
    mean, std = model.predict(candidates_8, return_std=True)
    ucb = mean + beta_8 * std

    index = np.argmax(ucb)
    query = candidates_8[index]

    svm_class = int(
        selected_svm_models[8].predict(query.reshape(1, -1))[0]
    )

    proposals_8[name] = {
        "query": query.copy(),
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index]),
        "nearest_distance": float(nearest_distance_8[index]),
        "svm_class": svm_class
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(
        f"Distance to nearest observation: "
        f"{nearest_distance_8[index]:.6f}"
    )
    print("SVM class (1 = above threshold):", svm_class)


Model: A_original
Proposed query: 0.085652-0.118248-0.325708-0.015452-0.990864-0.755098-0.045840-0.899890
Predicted output: 9.775451e+00
Predictive standard deviation: 3.154485e-01
UCB score: 1.040635e+01
Distance to nearest observation: 0.766722
SVM class (1 = above threshold): 1

Model: B_wider
Proposed query: 0.013017-0.039459-0.064531-0.040319-0.936756-0.624232-0.026406-0.945946
Predicted output: 9.840428e+00
Predictive standard deviation: 1.473795e-01
UCB score: 1.013519e+01
Distance to nearest observation: 0.641191
SVM class (1 = above threshold): 1


### Function 8 — UCB comparison

Results:

**A_original**
- query: `0.085652-0.118248-0.325708-0.015452-0.990864-0.755098-0.045840-0.899890`
- mean ≈ `9.775451`
- std ≈ `0.315449`
- UCB ≈ `10.40635`
- nearest distance ≈ `0.7667`
- SVM class = `1`

**B_wider**
- query: `0.013017-0.039459-0.064531-0.040319-0.936756-0.624232-0.026406-0.945946`
- mean ≈ `9.840428`
- std ≈ `0.147380`
- UCB ≈ `10.13519`
- nearest distance ≈ `0.6412`
- SVM class = `1`

A_original has the larger UCB because it is much more uncertain. B_wider has the higher predicted mean and much lower uncertainty.

Given the enormous improvement in B_wider's log-marginal likelihood, the lower UCB should not be interpreted as evidence that A_original is the better model. Instead, the original model is simply much less certain in this high-dimensional region.

Both proposals are highly exploratory and both are classified as above threshold by the SVM.


In [30]:
# Store the selected model and query for Function 8, round 3
selected_model_8 = models_8["B_wider"]
query_8_round3 = proposals_8["B_wider"]["query"].copy()

# Keep the prediction details for comparison with future feedback
selected_proposal_8_round3 = proposals_8["B_wider"].copy()

print(
    "Function 8 — round 3:",
    "-".join(f"{x:.6f}" for x in query_8_round3)
)

Function 8 — round 3: 0.013017-0.039459-0.064531-0.040319-0.936756-0.624232-0.026406-0.945946


### Function 8 — final Round 3 decision

The selected query is:

`0.013017-0.039459-0.064531-0.040319-0.936756-0.624232-0.026406-0.945946`

from **B_wider**.

The main reason is the dramatic improvement in GP fit. B_wider explains the accumulated observations far better while still proposing a high predicted response in a region that remains well separated from existing observations.


In [31]:
# Collect all selected queries for round 3
round_3_queries = {
    1: query_1_round3,
    2: query_2_round3,
    3: query_3_round3,
    4: query_4_round3,
    5: query_5_round3,
    6: query_6_round3,
    7: query_7_round3,
    8: query_8_round3
}

portal_queries_round3 = {}

for function_number, query in round_3_queries.items():
    query = np.round(np.asarray(query, dtype=float), 6)
    X = updated_data[function_number]["X"]

    # Validate dimensions, values and absence of previously evaluated points
    assert query.shape == (X.shape[1],)
    assert np.isfinite(query).all()
    assert np.all((query >= 0) & (query < 1))
    assert np.min(np.linalg.norm(X - query, axis=1)) > 1e-6

    portal_format = "-".join(f"{value:.6f}" for value in query)
    portal_queries_round3[function_number] = portal_format

    print(f"Function {function_number}: {portal_format}")

Function 1: 0.881159-0.530910
Function 2: 0.752835-0.997401
Function 3: 0.050178-0.066268-0.655743
Function 4: 0.368862-0.383152-0.454070-0.450029
Function 5: 0.505572-0.996280-0.970921-0.997435
Function 6: 0.546223-0.109744-0.472859-0.810424-0.001736
Function 7: 0.086618-0.151507-0.205694-0.787945-0.397953-0.790895
Function 8: 0.013017-0.039459-0.064531-0.040319-0.936756-0.624232-0.026406-0.945946


### Explanation — final Round 3 query list

This cell collects the eight selected queries into one final dictionary and prints them in the exact six-decimal format required by the portal.

The final Round 3 submissions are:

1. `0.881159-0.530910`
2. `0.752835-0.997401`
3. `0.050178-0.066268-0.655743`
4. `0.368862-0.383152-0.454070-0.450029`
5. `0.505572-0.996280-0.970921-0.997435`
6. `0.546223-0.109744-0.472859-0.810424-0.001736`
7. `0.086618-0.151507-0.205694-0.787945-0.397953-0.790895`
8. `0.013017-0.039459-0.064531-0.040319-0.936756-0.624232-0.026406-0.945946`

This consolidated list is the final check before copying the queries into the BBO portal.
